# I.23：把三边证书变成角相等证书
这是现代精确坐标教学重构，不是欧几里得圆规操作程序。原证先用 I.22 构造三边，再用 I.8（SSS）认定角。取源 C、D、E；目标 A、B 定义基射线。G 在 AB 射线上，F 在指定侧，AF=CD、AG=CE、FG=DE。

下面 6 个 Python 格由标准库 compile/exec 顺序执行并保存 stdout；没有启动 Jupyter kernel。仅有限案例检验，不证明一般定理。


In [ ]:
from geometry import construct, verify
from fractions import Fraction as Q
args=((0,0),(2,0),(1,2),(3,-1),(4,0),1)
c=construct(*args)
print('d:',c['radicand'])
print('F:',c['F'])
print('G:',c['G'])
print('verified:',verify(c,*args))


d: 5/2
F: {'constant': ['3', '-1'], 'sqrt_coefficient': ['-2/5', '6/5']}
G: {'constant': ['3', '-1'], 'sqrt_coefficient': ['1', '1']}
verified: True


## 精确表达与三边
点的含义为 constant + sqrt(radicand) × sqrt_coefficient。此例 d=5/2，G 的系数为 (1,1)，F 的系数为 (-2/5,6/5)。不把根号转成浮点。源三边平方为 4、5、5。


In [ ]:
print(c['checks'])
if [c['checks'][k] for k in ('AF2','AG2','FG2')]!=['4','5','5']:
    raise RuntimeError('three-side mismatch')
print('exact three-side identities passed')


{'AF2': '4', 'AG2': '5', 'FG2': '5', 'AF_dot_AG': '2', 'AG_cross_AF': '4'}
exact three-side identities passed


## 镜像不是失败
side=1 指 cross(AG,AF)>0，side=-1 为另一侧。源有向比较也取 cross(CE,CD)，不是 cross(CD,CE)。两支均复制普通内角；只有一支保持这个有向约定。


In [ ]:
mirror=construct(*args[:-1],-1)
for item in (c,mirror):
    print(item['side'],item['checks']['AG_cross_AF'],item['signed_orientation_preserved'])


1 4 False
-1 -4 True


## 严格证书及输入绑定
verify 重新构造规范 JSON 并比较，不是独立形式证明内核。布尔值 True 不得冒充整数 1，修改输入不得沿用旧证书。


In [ ]:
import copy
bad=copy.deepcopy(c);bad['side']=True
print('bool tamper:',verify(bad,*args))
changed=(*args[:2],(2,3),*args[3:])
print('different input:',verify(c,*changed))
if verify(bad,*args) or verify(c,*changed):
    raise RuntimeError('tampering accepted')


bool tamper: False
different input: False


## 退化域外输入
零角、平角、零射线不是本实现接受的非退化直线角。AB 也不能是零向量。禁止浮点和布尔坐标；可输入整数、有理数字符串、Fraction。


In [ ]:
for E in ((2,0),(-2,0),(0,0)):
    try:
        construct((0,0),(1,0),E,(0,0),(1,0))
    except ValueError as exc:
        print(E,str(exc))
    else:
        raise RuntimeError('degenerate angle accepted')


(2, 0) ordinary nondegenerate angle required
(-2, 0) ordinary nondegenerate angle required
(0, 0) zero ray


## 有限回归集
六组源形状、四组目标射线、三种源截点缩放、两侧支，共 144 证书。源截点改变会改变三角形尺寸，不改变相同两射线所夹普通角；证书仍须分别绑定输入。本程序只覆盖有理输入及单根式输出，不声称穷尽古典几何。


In [ ]:
from cases import run
print(run())


{'positive_certificates': 144, 'negative_inputs': 14, 'irrational_scale_certificates': 126, 'orientation_reversing_certificates': 72}
